##Read the file from Bronze

In [0]:
df2 =spark.table("codebasics.bronze.bronze_rating")

In [0]:
df2.limit(5).display()

user_id,anime_id,rating
45073,15583,7
45073,15609,6
45073,15689,9
45073,15699,7
45073,15751,6


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col 

##exploring the dataset : total rows

In [0]:
df2.count()

7813737

###total null value 

In [0]:
null_counts = df2.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df2.columns])
display(null_counts)

user_id,anime_id,rating
0,0,0


###check the duplicate values

In [0]:
duplicate_counts = df2.agg(*[F.count(df2[c]) - F.countDistinct(df2[c]) for c in df2.columns])
display(duplicate_counts)

(count(user_id) - count(DISTINCT user_id)),(count(anime_id) - count(DISTINCT anime_id)),(count(rating) - count(DISTINCT rating))
7740222,7802537,7813726


###Check rating distribution

In [0]:
rating_dist= df2.groupBy("rating").count().orderBy("rating").show()
display(rating_dist)

+------+-------+
|rating|  count|
+------+-------+
|    -1|1476496|
|     1|  16649|
|     2|  23150|
|     3|  41453|
|     4| 104291|
|     5| 282806|
|     6| 637775|
|     7|1375287|
|     8|1646019|
|     9|1254096|
|    10| 955715|
+------+-------+



###-1 means watched but not rated so replace -1 with 0 

In [0]:
df2 = df2.withColumn("rating", F.when(F.col("rating") == -1, 0).otherwise(F.col("rating")))

In [0]:
df2.display()

user_id,anime_id,rating
45073,15583,7
45073,15609,6
45073,15689,9
45073,15699,7
45073,15751,6
45073,15809,9
45073,15863,8
45073,15911,7
45073,16001,8
45073,16005,7


###check the rating column , is there any null value . if yes then how many

In [0]:
df2.select(
    F.count(F.when(F.col("rating").isNull(), 1)).alias("null_ratings")
).show()

+------------+
|null_ratings|
+------------+
|           0|
+------------+



### Cap the first letter of column name


In [0]:
RENAME_MAP = {
    "user_id": "User_id",
    "anime_id": "Anime_id",
    "rating": "User_rating"
}
for old_name, new_name in RENAME_MAP.items():
    df2 = df2.withColumnRenamed(old_name, new_name)

###print schema

In [0]:
df2.display()


User_id,Anime_id,User_rating
45073,15583,7
45073,15609,6
45073,15689,9
45073,15699,7
45073,15751,6
45073,15809,9
45073,15863,8
45073,15911,7
45073,16001,8
45073,16005,7


### write the silver rating file

In [0]:
df2.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("codebasics.silver.silver_rating")

In [0]:
df2.display()

User_id,Anime_id,User_rating
45073,15583,7
45073,15609,6
45073,15689,9
45073,15699,7
45073,15751,6
45073,15809,9
45073,15863,8
45073,15911,7
45073,16001,8
45073,16005,7
